# Priority Product Test Group Creation - Blocked Randomized Design (60 / 20 / 20)

## Objective

Assign **US-Domestic** OD markets to a **60% Control / 40% Treatment** design. The 40% treated markets are split evenly into two price arms:

- **Control** (60%) - no price change (multiplier 1.00); left unchanged but recorded.
- **T_minus20** (20%) - price multiplier 0.80 (-20%).
- **T_plus20** (20%) - price multiplier 1.20 (+20%).

The arms are balanced on market-level pre-treatment characteristics jointly:

- Priority Group
- Flight duration
- Day-of-week (DOW) demand profile
- Traveler-mix segment composition (business / bleisure / VFR / vacation / personal)
- PNR volume

## Design

Blocked randomization with a **block size of 5** (= 60 / 20 / 20 per block):

1. Build OD-level market features (Domestic only).
2. Log-transform PNR volume (it is highly skewed) and drop pax count (collinear with PNR).
3. Standardize the continuous features, then **whiten** them so distances are Mahalanobis.
4. Form blocks of 5 nearest-neighbor markets, **primarily within exact Priority Group x flight duration cells**.
5. Within each block, randomly assign **3 markets to Control, 1 to -20%, and 1 to +20%**.
6. Apply light rerandomization to select a balanced draw (Mahalanobis across the three arms + a volume-balance penalty **between the two treatment arms**).
7. Validate on OD counts, PG mix, flight-duration mix, DOW profile, traveler mix, PNR/PAX volume, and Mahalanobis distance.

## Notes on inference

Treatment is randomized within blocks (2 of every 5 look-alike markets), but the final draw is chosen by rerandomization. Downstream analysis should account for the blocked / rerandomized design or use covariate-adjusted modeling. Control is 60% of markets; each treatment arm is 20%.

## Outputs

- `priority_test_group_assignment_blocked` - full OD-to-arm assignment (Control / T_minus20 / T_plus20).
- `priority_pilot_plan_blocked` - combined deployment plan; Control rows are flagged and carry multiplier 1.00.

In [ ]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import itertools

RANDOM_SEED = 42

# Three arms: 90% Control, 5% -20%, 5% +20%.
CONTROL_LABEL = 'Control'
TREATMENT_ARMS = ['T_minus20', 'T_plus20']
ARMS = [CONTROL_LABEL] + TREATMENT_ARMS

TREATMENT_VALUE_MAP = {
    'Control': 1.00,
    'T_minus20': 0.80,
    'T_plus20': 1.20,
}

# Block size encodes the allocation: 18 Control + 1 (-20%) + 1 (+20%) per block
# of 20 => 90% / 5% / 5%. One treated market per treatment arm per block.
BLOCK_SIZE = 20
N_CONTROL_PER_BLOCK = BLOCK_SIZE - len(TREATMENT_ARMS)   # 18

DOW_COLS = [
    'pct_sun',
    'pct_mon',
    'pct_tue',
    'pct_wed',
    'pct_thu',
    'pct_fri',
    'pct_sat',
]

# OD-level traveler-mix composition (mean of the per-row segment probabilities).
# Included because traveler segment drives price elasticity, which the DOW
# profile only weakly proxies.
SEG_COLS = [
    'seg_business',
    'seg_bleisure',
    'seg_vfr',
    'seg_vacation',
    'seg_personal',
]

START_DATE = '2026-08-01'
END_DATE = '2026-08-14'

In [5]:
# Establish the Databricks Connect session for this (fresh) kernel.
# getOrCreate() with no prior spark.stop() is safe here - it just attaches.
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
print('Spark session ready. Test:', spark.range(3).count())

Spark session ready. Test: 3


In [ ]:
priority_airport = pd.read_excel('Priority_Group_Airport.xlsx')

priority_airport_spark = spark.createDataFrame(priority_airport)
priority_airport_spark.createOrReplaceTempView('priority_airport')

finalTransactionOfferSale = spark.table('rm_workspace.finalTransactionOfferSale_B')
finalTransactionOfferSale.createOrReplaceTempView('finalTransactionOfferSale')

print(f'finalTransactionOfferSale_B: {finalTransactionOfferSale.count():,} rows')

finalTransactionOfferSale_B: 52,582,746 rows


In [ ]:
itinerary = (
    spark.table('rm_workspace.tmp_pnr_spine')
    .filter(F.length(F.col('fare_basis_cd')) <= 8)
    .filter(F.col('pax_count') > 0)
)

itinerary.createOrReplaceTempView('itinerary')

In [ ]:
priority_airport_sdf = spark.table('priority_airport')
finalTransactionOfferSale = spark.table('finalTransactionOfferSale')

offer_sale = (
    finalTransactionOfferSale.alias('f')
    .join(
        priority_airport_sdf.alias('p'),
        F.col('f.od_origin') == F.col('p.Airport'),
        'left',
    )
    .withColumn('DOW', F.dayofweek('OD_dep_dt'))
    .withColumn('Priority_Group', F.coalesce(F.col('p.Group'), F.lit(6)))
)

offer_sale.createOrReplaceTempView('offer_sale')

In [ ]:
dow_counts = (
    itinerary
    .filter(F.col('region').like('%US48%'))
    .groupBy(
        F.col('od_origin_airprt_iata_cd').alias('od_origin'),
        F.col('od_destntn_airprt_iata_cd').alias('od_destination'),
        F.dayofweek('od_local_dep_dt').alias('dow'),
    )
    .agg(F.count('*').alias('dep_cnt'))
)

totals = (
    dow_counts
    .groupBy('od_origin', 'od_destination')
    .agg(F.sum('dep_cnt').alias('total_dep'))
)

od_dow = (
    dow_counts.alias('d')
    .join(totals.alias('t'), ['od_origin', 'od_destination'])
    .groupBy('od_origin', 'od_destination')
    .agg(
        F.round(100 * F.sum(F.when(F.col('dow') == 1, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_sun'),
        F.round(100 * F.sum(F.when(F.col('dow') == 2, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_mon'),
        F.round(100 * F.sum(F.when(F.col('dow') == 3, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_tue'),
        F.round(100 * F.sum(F.when(F.col('dow') == 4, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_wed'),
        F.round(100 * F.sum(F.when(F.col('dow') == 5, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_thu'),
        F.round(100 * F.sum(F.when(F.col('dow') == 6, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_fri'),
        F.round(100 * F.sum(F.when(F.col('dow') == 7, F.col('dep_cnt')).otherwise(0)) / F.max('total_dep'), 2).alias('pct_sat'),
    )
)

od_dow.createOrReplaceTempView('od_dow')

In [ ]:
pnr_od = (
    itinerary
    .filter(F.col('region').like('%US48%'))
    .groupBy(
        F.col('od_origin_airprt_iata_cd').alias('od_origin'),
        F.col('od_destntn_airprt_iata_cd').alias('od_destination'),
        'pnr_loctr_id',
    )
    .agg(F.max('pax_count').alias('pax_count'))
)

od_volume = (
    pnr_od
    .groupBy('od_origin', 'od_destination')
    .agg(
        F.countDistinct('pnr_loctr_id').alias('pnr_cnt'),
        F.sum('pax_count').alias('pax_cnt'),
    )
)

od_volume.createOrReplaceTempView('od_volume')

In [ ]:
od_market_features = (
    offer_sale.alias('o')
    .join(od_dow.alias('d'), ['od_origin', 'od_destination'], 'left')
    .join(od_volume.alias('v'), ['od_origin', 'od_destination'], 'left')
    .filter(F.col('region_group') == 'Domestic')
    .groupBy(
        'od_origin',
        'od_destination',
        'pct_sun',
        'pct_mon',
        'pct_tue',
        'pct_wed',
        'pct_thu',
        'pct_fri',
        'pct_sat',
        'pnr_cnt',
        'pax_cnt',
    )
    .agg(
        F.max('Priority_Group').alias('priority_group'),
        F.max('FlightDuration').alias('flight_duration'),
        F.avg('avg_business_prob').alias('seg_business'),
        F.avg('avg_bleisure_prob').alias('seg_bleisure'),
        F.avg('avg_vfr_prob').alias('seg_vfr'),
        F.avg('avg_vacation_prob').alias('seg_vacation'),
        F.avg('avg_personal_prob').alias('seg_personal'),
    )
    .withColumn('pnr_cnt', F.coalesce(F.col('pnr_cnt'), F.lit(0)))
    .withColumn('pax_cnt', F.coalesce(F.col('pax_cnt'), F.lit(0)))
)

od_market_features.createOrReplaceTempView('od_market_features')

In [ ]:
od_features = spark.sql('''
SELECT
    od_origin,
    od_destination,
    priority_group,
    flight_duration,
    pct_sun,
    pct_mon,
    pct_tue,
    pct_wed,
    pct_thu,
    pct_fri,
    pct_sat,
    pnr_cnt,
    pax_cnt,
    seg_business,
    seg_bleisure,
    seg_vfr,
    seg_vacation,
    seg_personal
FROM od_market_features
''').toPandas()

for c in DOW_COLS:
    od_features[c] = pd.to_numeric(od_features[c], errors='coerce')

for c in SEG_COLS:
    od_features[c] = pd.to_numeric(od_features[c], errors='coerce')

for c in ['pnr_cnt', 'pax_cnt']:
    od_features[c] = pd.to_numeric(od_features[c], errors='coerce').fillna(0)

od_features[DOW_COLS] = od_features[DOW_COLS].fillna(0)
od_features[SEG_COLS] = od_features[SEG_COLS].fillna(0)

print(f'OD markets: {len(od_features):,}')
display(od_features.head())

OD markets: 50,644


,od_origin,od_destination,priority_group,flight_duration,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat,pnr_cnt,pax_cnt,seg_business,seg_bleisure,seg_vfr,seg_vacation,seg_personal
0,ABE,MSO,5,Short,17.86,21.43,10.71,17.86,10.71,7.14,14.29,17,17,0.116772,0.054669,0.498361,0.258331,0.071867
1,ABI,SMF,3,Short,13.99,13.99,11.49,15.81,13.54,15.36,15.81,730,730,0.303742,0.068133,0.462136,0.081258,0.083795
2,ACT,BNA,3,Ultra_Short,16.27,17.71,12.89,10.24,15.78,14.34,12.77,716,716,0.363163,0.092644,0.357358,0.114649,0.072185
3,ACY,MLU,6,Short,0.00,0.00,50.00,0.00,0.00,50.00,0.00,2,2,0.296927,0.070150,0.360191,0.156109,0.116609
4,AEX,MLB,3,Ultra_Short,0.00,0.00,0.00,0.00,100.00,0.00,0.00,1,1,0.132829,0.029700,0.519171,0.112300,0.063143


## Feature preparation

Build the feature vector used for blocking and balance scoring. Volume is log-transformed because it is heavy-tailed; pax count is dropped as it is collinear with PNR count. The DOW demand profile and the traveler-mix segment shares (business / bleisure / VFR / vacation / personal) are added as continuous features. All standardized features are then **whitened**, so Euclidean distance on the whitened coordinates equals Mahalanobis distance - this keeps the correlated DOW and segment columns from dominating volume and drops their exact sum dependencies. Priority Group and flight duration are kept as exact-match keys rather than distances.

In [ ]:
# Continuous balancing features, standardized to a common scale.
# Volume is heavy-tailed, so use log1p. pax_cnt is dropped because it is
# almost perfectly collinear with pnr_cnt and would double-count volume.
# The DOW shares and the traveler-mix segment shares are each compositional
# (each set sums to a constant), so one reference column is dropped from each
# to remove the exact linear dependency. No information is lost - the dropped
# share is implied by the others - and it keeps the whitening well-posed
# (otherwise rounding noise in the near-null "sum" direction gets amplified).
od_features['log_pnr'] = np.log1p(od_features['pnr_cnt'])

DOW_BLOCK_COLS = DOW_COLS[:-1]   # drop pct_sat (implied by the other six)
SEG_BLOCK_COLS = SEG_COLS[:-1]   # drop seg_personal (implied by the other four)

CONT_FEATURES = ['log_pnr'] + DOW_BLOCK_COLS + SEG_BLOCK_COLS
Z_COLS = ['z_' + c for c in CONT_FEATURES]

mu = od_features[CONT_FEATURES].mean()
sd = od_features[CONT_FEATURES].std(ddof=0).replace(0, 1.0)
od_features[Z_COLS] = (od_features[CONT_FEATURES] - mu) / sd

# Whitening transform: Euclidean distance on the whitened coordinates equals
# Mahalanobis distance in the standardized space. This fixes the feature-
# weighting problem - without it the DOW and segment columns would swamp the
# single volume dimension. With the reference columns dropped above, the
# covariance is full rank, so no eigen-direction should need zeroing.
Zc = od_features[Z_COLS].to_numpy(dtype=float)
cov_z = np.cov(Zc, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov_z)
tol = 1e-8 * eigvals.max()
inv_sqrt = np.where(eigvals > tol, 1.0 / np.sqrt(eigvals), 0.0)
WHITEN = eigvecs @ np.diag(inv_sqrt) @ eigvecs.T
W_COLS = ['w_' + c for c in CONT_FEATURES]
od_features[W_COLS] = Zc @ WHITEN

# Categorical design factors are enforced as exact-match block keys, so every
# block is homogeneous on Priority Group and flight duration.
od_features['block_key'] = (
    od_features['priority_group'].astype(str)
    + '|'
    + od_features['flight_duration'].astype(str)
)

n_keys = od_features['block_key'].nunique()
print(f'Continuous features (standardized): {CONT_FEATURES}')
print(f'Effective whitened dimensions: {int((inv_sqrt > 0).sum())} of {len(CONT_FEATURES)}')
print(f'Exact-match keys (PG x duration): {n_keys}')

Continuous features (standardized): ['log_pnr', 'pct_sun', 'pct_mon', 'pct_tue', 'pct_wed', 'pct_thu', 'pct_fri', 'seg_business', 'seg_bleisure', 'seg_vfr', 'seg_vacation']
Effective whitened dimensions: 11 of 11
Exact-match keys (PG x duration): 30


## Blocking

Form blocks of **5** nearest-neighbor markets (by whitened / Mahalanobis distance) within each exact Priority Group x flight duration cell. Each block later contributes **3 Control markets, 1 at -20%, and 1 at +20%** (60 / 20 / 20). Markets that cannot fill a complete block within their cell fall into a remainder pool blocked separately, **without** the exact-match constraint; any final leftover is assigned to Control. The number of markets landing in mixed-key blocks is reported so the strength of the exact-match design is explicit.

In [ ]:
Wmat = od_features[W_COLS]


def form_blocks(features_df, dist_mat, block_key_col, k, seed):
    # Greedy nearest-neighbor blocking into blocks of size k, using Euclidean
    # distance on the whitened coordinates (equivalent to Mahalanobis).
    # Pass 1 blocks within each exact-match key; markets that cannot fill a full
    # block go to a remainder pool. Pass 2 blocks the remainder ignoring the
    # exact-match constraint. Any final leftover (< k) keeps a NaN block id.
    # Implemented on NumPy arrays with an alive-mask (no per-row pandas .loc),
    # so it scales to tens of thousands of markets in seconds.
    rng = np.random.default_rng(seed)
    Z = dist_mat.to_numpy(dtype=float)
    keys = features_df[block_key_col].to_numpy()
    n = len(features_df)
    block_of_pos = np.full(n, -1, dtype=np.int64)
    counter = {'next': 0}

    def greedy(pos_arr):
        # pos_arr holds global row positions. Form blocks of the k nearest
        # still-alive markets around a rolling seed. Returns leftover positions.
        pos = np.asarray(pos_arr, dtype=np.int64)
        rng.shuffle(pos)
        Zsub = Z[pos]
        alive = np.ones(len(pos), dtype=bool)
        ptr = 0
        n_alive = len(pos)
        while n_alive >= k:
            while not alive[ptr]:
                ptr += 1
            alive_idx = np.flatnonzero(alive)
            d = np.sqrt(((Zsub[alive_idx] - Zsub[ptr]) ** 2).sum(axis=1))
            nearest = alive_idx[np.argsort(d)[:k]]
            block_of_pos[pos[nearest]] = counter['next']
            counter['next'] += 1
            alive[nearest] = False
            n_alive -= k
        return pos[alive].tolist()

    remainder = []
    for key in pd.unique(keys):
        remainder.extend(greedy(np.flatnonzero(keys == key)))

    final_leftover_pos = greedy(remainder)

    block_id = pd.Series(
        np.where(block_of_pos >= 0, block_of_pos, np.nan),
        index=features_df.index,
    )
    final_leftover = features_df.index[final_leftover_pos].tolist()
    return block_id, final_leftover


block_id, leftover = form_blocks(od_features, Wmat, 'block_key', BLOCK_SIZE, RANDOM_SEED)
od_features['block_id'] = block_id

n_blocks = int(od_features['block_id'].notna().sum() // BLOCK_SIZE)
print(f'Full blocks of {BLOCK_SIZE} formed: {n_blocks} (covering {n_blocks * BLOCK_SIZE:,} markets)')
print(f'Treated markets: {n_blocks * len(TREATMENT_ARMS):,} '
      f'({100 * n_blocks * len(TREATMENT_ARMS) / len(od_features):.1f}% of all markets)')
print(f'Leftover markets (assigned to Control): {len(leftover)}')

# Diagnostic: how many blocks mix Priority Group x duration keys (these can only
# arise in the remainder pass, which drops the exact-match constraint). If this
# is tiny, the exact-match design effectively holds for all markets.
blocked = od_features.dropna(subset=['block_id'])
keys_per_block = blocked.groupby('block_id')['block_key'].nunique()
mixed_blocks = keys_per_block[keys_per_block > 1]
markets_in_mixed = int(od_features['block_id'].isin(mixed_blocks.index).sum())
print(f'Blocks mixing PG x duration keys: {len(mixed_blocks):,}')
print(f'Markets in mixed blocks: {markets_in_mixed:,} '
      f'({100 * markets_in_mixed / len(od_features):.1f}% of all markets)')

Full blocks of 5 formed: 10128 (covering 50,640 markets)
Treated markets: 20,256 (40.0% of all markets)
Leftover markets (assigned to Control): 4
Blocks mixing PG x duration keys: 11
Markets in mixed blocks: 55 (0.1% of all markets)


## Balance metric

Overall imbalance is the mean pairwise Mahalanobis distance between the arm mean-vectors, using the market-level covariance (pseudo-inverse, so collinear features are handled). The blocking step above uses the same whitened geometry, so the design and the diagnostic are consistent. The inverse covariance does not depend on arm labels, so it is computed once (`VI_GLOBAL`) and reused across rerandomization trials.

In [ ]:
def compute_VI(df, feature_cols):
    # Inverse covariance (pseudo-inverse) of the market-level features. This
    # does not depend on arm labels, so it is computed once and reused.
    X = df[feature_cols].to_numpy(dtype=float)
    cov = np.cov(X, rowvar=False)
    return np.linalg.pinv(cov)


def arm_balance_mahalanobis(df, feature_cols, VI=None, group_col='test_group'):
    # Overall imbalance = mean pairwise Mahalanobis distance between arm
    # mean-vectors. Scale-free and correlation-aware, so mixed units and
    # collinear features are handled automatically. Lower is better.
    # Pass a precomputed VI to avoid recomputing the covariance every call.
    if VI is None:
        VI = compute_VI(df, feature_cols)
    means = df.groupby(group_col)[feature_cols].mean()
    arms = list(means.index)
    dists = {}
    for a, b in itertools.combinations(arms, 2):
        diff = (means.loc[a] - means.loc[b]).to_numpy()
        dists[a + ' vs ' + b] = float(np.sqrt(diff @ VI @ diff))
    overall = float(np.mean(list(dists.values())))
    return overall, pd.Series(dists)


# Covariance is invariant to arm labels, so precompute the inverse once.
VI_GLOBAL = compute_VI(od_features, Z_COLS)

## Assignment and rerandomization

Within each block of 5, randomly pick 2 markets for the treatments (one -20%, one +20%) and assign the other 3 to Control. Repeat the random draw many times and keep the assignment that minimizes a combined objective: the mean pairwise Mahalanobis imbalance across the three arms **plus** a penalty on the total-PNR gap **between the two treatment arms** (control carries ~3x each treatment arm's volume by design, so only the treatment arms are compared on aggregate volume). Leftover markets (fewer than a full block) go to Control.

In [ ]:
# Vectorized rerandomization for the 90 / 5 / 5 design.
# Codes: 0 = Control, 1 = T_minus20, 2 = T_plus20.
# Within each block of BLOCK_SIZE, exactly one market goes to each treatment arm
# and the remaining BLOCK_SIZE - 2 go to Control. Leftover markets (that never
# formed a full block) stay Control.
Zvals = od_features[Z_COLS].to_numpy(dtype=float)
pnr = od_features['pnr_cnt'].to_numpy(dtype=float)
n_rows = len(od_features)
positions = np.arange(n_rows)

full_mask = od_features['block_id'].notna().to_numpy()
full_pos = positions[full_mask]
full_blk = od_features['block_id'].to_numpy()[full_mask]

# Sort full markets by block id so each consecutive BLOCK_SIZE positions is one
# block (every full block has exactly BLOCK_SIZE members by construction).
order = np.argsort(full_blk, kind='stable')
block_member_pos = full_pos[order].reshape(-1, BLOCK_SIZE)
leftover_pos = positions[~full_mask]

n_blocks_full = block_member_pos.shape[0]
block_rows = np.arange(n_blocks_full)

N_GROUPS = len(ARMS)   # 3: Control, T_minus20, T_plus20
pair_a, pair_b = zip(*itertools.combinations(range(N_GROUPS), 2))
pair_a = np.array(pair_a)
pair_b = np.array(pair_b)

# Weight on treatment-arm volume balance vs the Mahalanobis score.
VOLUME_WEIGHT = 1.0


def build_codes(seed):
    # Everything Control (0), then place one -20% (1) and one +20% (2) per block.
    rng = np.random.default_rng(seed)
    codes = np.zeros(n_rows, dtype=np.int64)
    picks = np.argsort(rng.random((n_blocks_full, BLOCK_SIZE)), axis=1)[:, :2]
    codes[block_member_pos[block_rows, picks[:, 0]]] = 1   # T_minus20
    codes[block_member_pos[block_rows, picks[:, 1]]] = 2   # T_plus20
    return codes


def score_components(codes):
    # Mean pairwise Mahalanobis distance across the three arm mean-vectors.
    means = np.vstack([Zvals[codes == a].mean(axis=0) for a in range(N_GROUPS)])
    diffs = means[pair_a] - means[pair_b]
    maha = np.sqrt(np.einsum('ij,jk,ik->i', diffs, VI_GLOBAL, diffs)).mean()
    # Volume balance BETWEEN the two treatment arms (control is ~18x by design).
    pnr_minus = pnr[codes == 1].sum()
    pnr_plus = pnr[codes == 2].sum()
    vol_gap = abs(pnr_minus - pnr_plus) / ((pnr_minus + pnr_plus) / 2)
    return maha, vol_gap


def balance_score(codes):
    maha, vol_gap = score_components(codes)
    return maha + VOLUME_WEIGHT * vol_gap


R = 2000
best_score = np.inf
best_codes = None
best_trial = None

for r in range(R):
    codes = build_codes(RANDOM_SEED + r)
    s = balance_score(codes)
    if s < best_score:
        best_score = s
        best_codes = codes
        best_trial = r

label_arr = np.array(ARMS, dtype=object)   # 0->Control, 1->T_minus20, 2->T_plus20
od_features['test_group'] = label_arr[best_codes]

best_maha, best_vol_gap = score_components(best_codes)
counts = od_features['test_group'].value_counts()
print(f'Best combined score: {best_score:.4f} (trial {best_trial} of {R})')
print(f'  Mahalanobis imbalance (3 arms): {best_maha:.4f}')
print(f'  Treatment-arm PNR gap:          {100 * best_vol_gap:.2f}% of treatment mean')
print('  Arm sizes: ' + ', '.join(f'{a}={int(counts.get(a, 0)):,}' for a in ARMS))

Best combined score: 0.0107 (trial 478 of 2000)
  Mahalanobis imbalance (3 arms): 0.0105
  Treatment-arm PNR gap:          0.02% of treatment mean
  Arm sizes: Control=30,388, T_minus20=10,128, T_plus20=10,128


In [ ]:
def validate_assignment(df, title='BALANCE SUMMARY'):
    print('=' * 70)
    print(title)
    print('=' * 70)

    count_balance = df.groupby('test_group').size()
    print('\nOD count by group (90 / 5 / 5 by design):')
    print(count_balance)

    pg_balance = pd.crosstab(df['test_group'], df['priority_group'], normalize='index') * 100
    print('\nPriority Group mix:')
    display(pg_balance.round(2))
    print('\nPriority Group max-min percentage point imbalance:')
    print((pg_balance.max() - pg_balance.min()).round(3))

    fd_balance = pd.crosstab(df['test_group'], df['flight_duration'], normalize='index') * 100
    print('\nFlight Duration mix:')
    display(fd_balance.round(2))
    print('\nFlight Duration max-min percentage point imbalance:')
    print((fd_balance.max() - fd_balance.min()).round(3))

    dow_balance = df.groupby('test_group')[DOW_COLS].mean()
    print('\nDOW profile:')
    display(dow_balance.round(2))
    print('\nDOW max-min percentage point imbalance:')
    print((dow_balance.max() - dow_balance.min()).round(3))

    seg_balance = df.groupby('test_group')[SEG_COLS].mean()
    print('\nTraveler-mix segment profile (mean probability):')
    display(seg_balance.round(3))
    print('\nTraveler-mix max-min imbalance:')
    print((seg_balance.max() - seg_balance.min()).round(4))

    volume_summary = (
        df.groupby('test_group')
        .agg(
            n_ods=('od_origin', 'size'),
            total_pnrs=('pnr_cnt', 'sum'),
            avg_pnrs_per_od=('pnr_cnt', 'mean'),
            total_pax=('pax_cnt', 'sum'),
            avg_pax_per_od=('pax_cnt', 'mean'),
        )
        .round(2)
    )
    print('\nVolume summary:')
    display(volume_summary)

    # Total volume per arm is ~18:1:1 by design; the balance that matters is
    # between the two treatment arms, plus per-market averages across all arms.
    treat = [a for a in TREATMENT_ARMS if a in volume_summary.index]
    if len(treat) == 2:
        tmin = volume_summary.loc[treat[0], 'total_pnrs']
        tplus = volume_summary.loc[treat[1], 'total_pnrs']
        gap = abs(tmin - tplus)
        gap_pct = 100 * gap / ((tmin + tplus) / 2)
        print(f'\nTreatment-arm PNR gap ({treat[0]} vs {treat[1]}):')
        print(f'{gap:,.0f} PNRs ({gap_pct:.2f}% of treatment mean)')

    print('\nAvg PNRs per OD (should match across arms if treated is representative):')
    print(volume_summary['avg_pnrs_per_od'])

    return {
        'count_balance': count_balance,
        'pg_balance': pg_balance,
        'fd_balance': fd_balance,
        'dow_balance': dow_balance,
        'seg_balance': seg_balance,
        'volume_summary': volume_summary,
    }


blocked_validation = validate_assignment(od_features, title='BLOCKED ASSIGNMENT BALANCE SUMMARY')

overall, pair_dists = arm_balance_mahalanobis(od_features, Z_COLS, VI=VI_GLOBAL)
print('\nMahalanobis imbalance (overall mean pairwise):', round(overall, 4))
print('\nPairwise Mahalanobis distances:')
display(pair_dists.round(4))

print('\nPer-arm standardized feature means (closer to each other = better):')
display(od_features.groupby('test_group')[Z_COLS].mean().round(3))

BLOCKED ASSIGNMENT BALANCE SUMMARY

OD count by group (60 / 20 / 20 by design):
test_group
Control      30388
T_minus20    10128
T_plus20     10128
dtype: int64



Priority Group mix:


priority_group,1,2,3,4,5,6
test_group,,,,,,
Control,3.73,16.34,34.19,28.42,7.29,10.04
T_minus20,3.72,16.34,34.18,28.43,7.29,10.04
T_plus20,3.72,16.34,34.19,28.42,7.30,10.03



Priority Group max-min percentage point imbalance:
priority_group
1    0.006
2    0.001
3    0.010
4    0.010
5    0.011
6    0.010
dtype: float64

Flight Duration mix:


flight_duration,Medium,Short,True_Long,Ultra_Short
test_group,,,,
Control,10.65,38.85,0.11,50.40
T_minus20,10.64,38.84,0.12,50.40
T_plus20,10.66,38.85,0.11,50.39


...

,n_ods,total_pnrs,avg_pnrs_per_od,total_pax,avg_pax_per_od
test_group,,,,,
Control,30388,35228441,1159.29,35228845,1159.30
T_minus20,10128,12060447,1190.80,12060574,1190.81
T_plus20,10128,12062367,1190.99,12062511,1191.01



Treatment-arm PNR gap (T_minus20 vs T_plus20):
1,920 PNRs (0.02% of treatment mean)

Avg PNRs per OD (should match across arms if treated is representative):
test_group
Control      1159.29
T_minus20    1190.80
T_plus20     1190.99
Name: avg_pnrs_per_od, dtype: float64

Mahalanobis imbalance (overall mean pairwise): 0.0105

Pairwise Mahalanobis distances:


Control vs T_minus20     0.0088
Control vs T_plus20      0.0098
T_minus20 vs T_plus20    0.0129
dtype: float64


Per-arm standardized feature means (closer to each other = better):


,z_log_pnr,z_pct_sun,z_pct_mon,z_pct_tue,z_pct_wed,z_pct_thu,z_pct_fri,z_seg_business,z_seg_bleisure,z_seg_vfr,z_seg_vacation
test_group,,,,,,,,,,,
Control,-0.001,-0.001,-0.000,-0.000,0.001,0.000,-0.002,-0.001,-0.000,0.001,0.001
T_minus20,0.000,0.001,-0.002,0.000,-0.003,-0.003,0.004,-0.000,0.001,0.001,0.001
T_plus20,0.002,0.003,0.004,0.001,0.000,0.002,0.001,0.004,0.000,-0.003,-0.004


## Output

Register the blocked assignment as a temp view. The table write is left commented out, mirroring the original notebook.

In [ ]:
final_assignment = od_features[
    [
        'od_origin',
        'od_destination',
        'priority_group',
        'flight_duration',
        'test_group',
        'pnr_cnt',
        'pax_cnt',
    ]
].copy()

final_assignment_spark = spark.createDataFrame(final_assignment)
final_assignment_spark.createOrReplaceTempView('priority_test_group_assignment_blocked')

# final_assignment_spark.write.mode('overwrite').saveAsTable(
#     'rm_workspace.priority_test_group_assignment_blocked'
# )

## Deployment plan

Build a single **combined** plan over all markets. Treated markets carry their price multiplier in `Result`; **Control markets are kept for recording**, clearly flagged via the `arm` column and `Comment`, with `Result` = 1.00. Each row is formatted in the pricing-tool layout (a `Loc1` origin/destination pair and travel-date window). Registered as `priority_pilot_plan_blocked`.

In [ ]:
# All markets active for the full test window.
plan = od_features.copy()
plan['Travel dates (Start)'] = START_DATE
plan['Travel dates (End)'] = END_DATE

plan['Loc1'] = 'P:' + plan['od_origin'].astype(str)
plan['Loc2'] = 'P:' + plan['od_destination'].astype(str)
plan['Sector direction'] = 'From Loc1 to Loc2'

# Clearly differentiate Control (recorded, no change) from the treated arms.
plan['arm'] = plan['test_group']
plan['Comment'] = np.where(
    plan['test_group'] == CONTROL_LABEL,
    'Control - no change',
    'Random Testing',
)
plan['Segment matches required'] = ''
plan['Result'] = plan['test_group'].map(TREATMENT_VALUE_MAP)

priority_plan = plan[
    [
        'arm',
        'Comment',
        'Segment matches required',
        'Travel dates (Start)',
        'Travel dates (End)',
        'Loc1',
        'Loc2',
        'Sector direction',
        'Result',
    ]
].copy()

priority_plan['Travel dates (Start)'] = pd.to_datetime(priority_plan['Travel dates (Start)']).dt.strftime('%m/%d/%Y')
priority_plan['Travel dates (End)'] = pd.to_datetime(priority_plan['Travel dates (End)']).dt.strftime('%m/%d/%Y')

# Split into separate treatment and control exports.
priority_plan_treatment = priority_plan[priority_plan['arm'] != CONTROL_LABEL].copy()
priority_plan_control = priority_plan[priority_plan['arm'] == CONTROL_LABEL].copy()

print(f'Treatment rows: {len(priority_plan_treatment):,}')
display(priority_plan_treatment.head(10))
print(f'\nControl rows: {len(priority_plan_control):,}')
display(priority_plan_control.head(5))

In [ ]:
# Separate temp views for treatment and control.
treatment_spark = spark.createDataFrame(priority_plan_treatment)
treatment_spark.createOrReplaceTempView('priority_pilot_plan_treatment')

control_spark = spark.createDataFrame(priority_plan_control)
control_spark.createOrReplaceTempView('priority_pilot_plan_control')

# treatment_spark.write.mode('overwrite').saveAsTable(
#     'rm_workspace.priority_pilot_plan_treatment'
# )
# control_spark.write.mode('overwrite').saveAsTable(
#     'rm_workspace.priority_pilot_plan_control'
# )

In [ ]:
priority_template = pd.read_csv("Priority_Test_Template.csv")
priority_template


In [ ]:
# Export treatment and control plans to CSV (workspace folder).
output_dir = '/Workspace/Users/939510@corpaa.aa.com/KRATOS_Exploration'

priority_plan_treatment.to_csv(
    f'{output_dir}/priority_pilot_plan_treatment.csv',
    index=False,
)
priority_plan_control.to_csv(
    f'{output_dir}/priority_pilot_plan_control.csv',
    index=False,
)

print(f'Treatment exported: {len(priority_plan_treatment):,} rows')
print(f'Control exported:   {len(priority_plan_control):,} rows')
print(f'\nFiles saved to: {output_dir}/')

In [ ]:
# Illustration: show a few complete blocks. Each block = 5 look-alike markets
# that share the same Priority Group x flight duration, one handed to each arm.
show_cols = [
    'block_id', 'test_group', 'od_origin', 'od_destination',
    'priority_group', 'flight_duration',
    'pnr_cnt', 'pct_mon', 'pct_sat', 'seg_business', 'seg_vacation',
]

pure = od_features.dropna(subset=['block_id'])
example_ids = pure['block_id'].drop_duplicates().head(3).tolist()

for bid in example_ids:
    blk = pure[pure['block_id'] == bid].sort_values('test_group')
    print(f'--- BLOCK {int(bid)} '
          f'(key = {blk["block_key"].iloc[0]}) ---')
    display(blk[show_cols])

--- BLOCK 73 (key = 5|Short) ---


,block_id,test_group,od_origin,od_destination,priority_group,flight_duration,pnr_cnt,pct_mon,pct_sat,seg_business,seg_vacation
0,73.0,Control,ABE,MSO,5,Short,17,21.43,14.29,0.116772,0.258331
25686,73.0,Control,FLG,PIA,5,Short,45,7.02,14.04,0.240141,0.233485
29572,73.0,Control,MLB,MKE,5,Short,130,23.13,12.50,0.217560,0.222235
7276,73.0,T_minus20,TRI,SGU,5,Short,62,16.05,16.05,0.187543,0.228329
5188,73.0,T_plus20,MLB,ATW,5,Short,34,13.64,15.91,0.184017,0.243279


--- BLOCK 1264 (key = 3|Short) ---


,block_id,test_group,od_origin,od_destination,priority_group,flight_duration,pnr_cnt,pct_mon,pct_sat,seg_business,seg_vacation
1,1264.0,Control,ABI,SMF,3,Short,730,13.99,15.81,0.303742,0.081258
27284,1264.0,Control,ORF,MFE,3,Short,520,10.53,15.40,0.271020,0.102292
34159,1264.0,Control,TUS,GSP,3,Short,785,12.80,13.29,0.257720,0.112772
42153,1264.0,T_minus20,ONT,FSM,3,Short,486,10.36,17.15,0.292886,0.076218
24148,1264.0,T_plus20,MAF,FAT,3,Short,445,10.33,12.61,0.244787,0.106641


--- BLOCK 1830 (key = 3|Ultra_Short) ---


,block_id,test_group,od_origin,od_destination,priority_group,flight_duration,pnr_cnt,pct_mon,pct_sat,seg_business,seg_vacation
10711,1830.0,Control,LCH,BNA,3,Ultra_Short,759,17.16,6.49,0.345526,0.142430
37177,1830.0,Control,CRP,ABQ,3,Ultra_Short,456,14.13,7.69,0.355342,0.146777
43659,1830.0,Control,LFT,BNA,3,Ultra_Short,1071,16.31,5.59,0.363972,0.142694
2540,1830.0,T_minus20,COU,TPA,3,Ultra_Short,725,14.83,10.90,0.354149,0.154638
2,1830.0,T_plus20,ACT,BNA,3,Ultra_Short,716,17.71,12.77,0.363163,0.114649


In [22]:
# DOW examples vs trip intent (traveler segment).
# Restrict to reasonably busy markets so the DOW profile is not just noise from
# a handful of departures.
ex = od_features[od_features['pnr_cnt'] >= 500].copy()
ex['weekday_share'] = ex[['pct_mon', 'pct_tue', 'pct_wed', 'pct_thu']].sum(axis=1)
ex['weekend_share'] = ex[['pct_fri', 'pct_sat', 'pct_sun']].sum(axis=1)

cols = [
    'od_origin', 'od_destination', 'pnr_cnt',
    'pct_sun', 'pct_mon', 'pct_tue', 'pct_wed', 'pct_thu', 'pct_fri', 'pct_sat',
    'weekday_share', 'weekend_share',
    'seg_business', 'seg_vacation', 'seg_vfr',
]

print('=== MOST WEEKDAY-HEAVY markets (expect business-leaning trip intent) ===')
display(ex.sort_values('weekday_share', ascending=False).head(6)[cols].round(2))

print('=== MOST WEEKEND-HEAVY markets (expect leisure/vacation-leaning) ===')
display(ex.sort_values('weekend_share', ascending=False).head(6)[cols].round(2))

=== MOST WEEKDAY-HEAVY markets (expect business-leaning trip intent) ===


,od_origin,od_destination,pnr_cnt,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat,weekday_share,weekend_share,seg_business,seg_vacation,seg_vfr
15394,LFT,SBA,598,4.12,31.54,44.53,7.13,3.80,5.39,3.49,87.00,13.00,0.73,0.04,0.10
34560,MAF,AEX,918,3.68,11.59,24.92,36.40,13.76,5.09,4.55,86.67,13.32,0.83,0.01,0.08
15468,SBA,LFT,587,3.42,5.21,12.87,59.93,7.98,7.17,3.42,85.99,14.01,0.74,0.04,0.10
42369,LCH,MAF,950,9.52,14.70,34.27,19.46,14.60,4.66,2.80,83.03,16.98,0.71,0.02,0.16
40555,MAF,LCH,816,4.67,8.97,23.09,26.67,23.92,9.45,3.23,82.65,17.35,0.70,0.02,0.16
38113,AEX,MAF,877,9.13,19.95,33.60,19.84,9.13,4.06,4.28,82.52,17.47,0.83,0.01,0.08


=== MOST WEEKEND-HEAVY markets (expect leisure/vacation-leaning) ===


,od_origin,od_destination,pnr_cnt,pct_sun,pct_mon,pct_tue,pct_wed,pct_thu,pct_fri,pct_sat,weekday_share,weekend_share,seg_business,seg_vacation,seg_vfr
2840,FCA,LGA,725,8.91,6.90,3.88,3.49,2.64,3.88,70.31,16.91,83.10,0.05,0.63,0.23
7180,LGA,FCA,927,6.70,6.14,5.46,6.14,8.80,7.56,59.21,26.54,73.47,0.05,0.62,0.24
29627,SAV,LGA,1670,38.22,10.12,7.04,5.67,7.17,14.43,17.34,30.00,69.99,0.19,0.26,0.37
20693,BZN,LGA,877,14.80,12.90,7.50,5.47,5.40,6.73,47.19,31.27,68.72,0.09,0.54,0.25
42416,MVY,ORD,560,14.49,10.45,8.43,6.74,6.29,7.19,46.40,31.91,68.08,0.10,0.56,0.22
42741,ACK,LGA,1340,49.56,13.93,7.54,6.91,5.57,5.69,10.80,33.95,66.05,0.09,0.52,0.27
